# Data Preprocessing and Training Data Creation

## Overview

This notebook processes the training data and prepares it for model training.
Per DATA.md:
- Training data (~1,500 samples) has translations aligned at document level
- Test data (~4,000 sentences) has sentence-level translations
- Various text preprocessing steps needed per the formatting suggestions

### Data Preprocessing Notes from DATA.md:
- Training data uses Ḫ ḫ characters while test data uses H h - substitution needed
- Remove/modern scribal notations: ! ? / : < > ˹ ˺ [ ]
- Replace breaks: [x] → <gap>, … → <big_gap>
- Line numbers are str type with values like 1, 1', or 1''

### External Resources
This notebook can create augmented data using:
- Akkademia Dataset (Gutherz et al. 2023): https://github.com/gaigutherz/Akkademia
- Sentence-level translations from Sentences_Oare_FirstWord_LinNum.csv

## 0. Text Preprocessing Functions

Per DATA.md formatting suggestions:
- Training data uses Ḫ ḫ characters but test data uses H h - substitution needed
- Various modern scribal notations to handle

In [ ]:
# Text preprocessing functions based on DATA.md formatting suggestions
import pandas as pd
import numpy as np
import re
import os

def preprocess_transliteration(text):
    """Preprocess Akkadian transliteration text per DATA.md formatting suggestions."""
    if pd.isna(text):
        return ""
    text = str(text)
    # Replace breaks
    text = re.sub(r'\[x\]', '<gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\[…\]', '<big_gap>', text)
    text = re.sub(r'\[…\]', '<big_gap>', text)
    # Remove modern scribal notations
    text = re.sub(r'[!?/:]', '', text)
    text = re.sub(r'[˹˺]', '', text)
    text = re.sub(r'\[|\]', '', text)
    # Note: Ḫ ḫ -> H h substitution may be needed for training data
    return text.strip()

def preprocess_translation(text):
    """Preprocess English translation text."""
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove modern notations
    text = re.sub(r'<|>', '', text)
    return text.strip()

print("Loading datasets...")
train = pd.read_csv('train.csv')
sentences = pd.read_csv('Sentences_Oare_FirstWord_LinNum.csv')
published = pd.read_csv('published_texts.csv')

print(f"Original train size: {len(train)}")
print(f"Sentences size: {len(sentences)}")
print(f"Published texts size: {len(published)}")

## 1. Understanding the Data Structure

In [ ]:
# Examine sentences data
print("--- Sentences Sample ---")
print(sentences[['display_name', 'translation', 'first_word_transcription']].head(10))

print("\n--- Published Texts Aliases Sample ---")
print(published['aliases'].head(20))

## 2. Extract Publication IDs from Sentences

In [ ]:
# Extract publication ID from display_name
# Example: "(Kayseri 22)" -> "Kayseri 22"
def extract_id(name):
    if pd.isna(name):
        return None
    match = re.search(r'\(([^)]+)\)', str(name))
    if match:
        return match.group(1).strip()
    return None

sentences['pub_id'] = sentences['display_name'].apply(extract_id)

print("Extracted publication IDs:")
print(sentences['pub_id'].value_counts().head(20))

## 3. Create Alias to Transliteration Mapping

In [ ]:
# Create mapping from aliases to transliteration
# Multiple aliases can map to the same text (separated by |)
alias_to_translit = {}
for idx, row in published.iterrows():
    if pd.notna(row['aliases']):
        for alias in row['aliases'].split('|'):
            alias_to_translit[alias.strip()] = row['transliteration']

print(f"Created mapping for {len(alias_to_translit)} aliases")

# Show some example mappings
print("\nExample aliases:")
for i, (alias, translit) in enumerate(list(alias_to_translit.items())[:5]):
    print(f"  {alias}: {translit[:50]}...")

## 4. Match Sentences to Transliterations

In [ ]:
# Match sentences to transliterations
def find_translit(pub_id):
    if pd.isna(pub_id):
        return None
    # Try exact match
    if pub_id in alias_to_translit:
        return alias_to_translit[pub_id]
    # Try without the first word (e.g., 'Kayseri 22' -> '22')
    parts = pub_id.split()
    if len(parts) > 1:
        short_id = ' '.join(parts[1:])
        if short_id in alias_to_translit:
            return alias_to_translit[short_id]
    return None

sentences['transliteration'] = sentences['pub_id'].apply(find_translit)

print(f"Matched transliterations: {sentences['transliteration'].notna().sum()}")
print(f"Unmatched: {sentences['transliteration'].isna().sum()}")

## 5. Filter and Clean Matched Data

In [ ]:
# Filter to matched sentences with both translation and transliteration
matched_sentences = sentences[
    (sentences['translation'].notna()) & 
    (sentences['transliteration'].notna()) &
    (sentences['translation'] != '') &
    (sentences['transliteration'] != '')
].copy()

print(f"Matched sentences with both translation and transliteration: {len(matched_sentences)}")

# Show sample matched data
print("\n--- Sample Matched Data ---")
for idx, row in matched_sentences.head(5).iterrows():
    print(f"\nID: {row['pub_id']}")
    print(f"Translation: {row['translation'][:100]}...")
    print(f"Transliteration: {row['transliteration'][:100]}...")

## 6. Remove Duplicates with Original Training Data

In [ ]:
# Check for duplicates in training data
existing_pairs = set(zip(train['transliteration'], train['translation']))

# Filter out sentences that already exist in training data
new_sentences = []
for idx, row in matched_sentences.iterrows():
    pair = (row['transliteration'], row['translation'])
    if pair not in existing_pairs:
        new_sentences.append(row)

print(f"New sentences not in original training data: {len(new_sentences)}")

## 7. Load External Akkademia Dataset

In [ ]:
# Load Akkademia data (if available)
akkademia_data = []

try:
    with open('akkademia_train.ak', 'r', encoding='utf-8') as f:
        akk_lines = f.readlines()
    with open('akkademia_train.en', 'r', encoding='utf-8') as f:
        en_lines = f.readlines()
    
    for akk, eng in zip(akk_lines, en_lines):
        akk = akk.strip()
        eng = eng.strip()
        if len(akk) > 10 and len(eng) > 10:
            akkademia_data.append({
                'transliteration': akk,
                'translation': eng,
                'source': 'akkademia'
            })
    
    print(f"Loaded Akkademia training data: {len(akkademia_data)} samples")
except FileNotFoundError:
    print("Akkademia data not found. Run augment_dataset.py first.")

try:
    with open('akkademia_valid.ak', 'r', encoding='utf-8') as f:
        akk_lines = f.readlines()
    with open('akkademia_valid.en', 'r', encoding='utf-8') as f:
        en_lines = f.readlines()
    
    for akk, eng in zip(akk_lines, en_lines):
        akk = akk.strip()
        eng = eng.strip()
        if len(akk) > 10 and len(eng) > 10:
            akkademia_data.append({
                'transliteration': akk,
                'translation': eng,
                'source': 'akkademia_valid'
            })
    
    print(f"Loaded Akkademia validation data: {len([x for x in akkademia_data if x['source'] == 'akkademia_valid'])} samples")
except FileNotFoundError:
    print("Akkademia validation data not found.")

## 8. Create Fully Augmented Training Dataset

In [ ]:
# Create training dataframe from original data
combined_train = train[['transliteration', 'translation']].copy()
combined_train['source'] = 'original'

# Add new sentences from supplemental data
new_train_df = pd.DataFrame(new_sentences)
if len(new_train_df) > 0:
    new_train_df = new_train_df[['transliteration', 'translation']].copy()
    new_train_df['source'] = 'sentences'
    combined_train = pd.concat([combined_train, new_train_df], ignore_index=True)

# Add Akkademia data
if akkademia_data:
    akkademia_df = pd.DataFrame(akkademia_data)
    combined_train = pd.concat([combined_train, akkademia_df], ignore_index=True)

# Remove duplicates
before_dedup = len(combined_train)
combined_train = combined_train.drop_duplicates(subset=['transliteration', 'translation'])
after_dedup = len(combined_train)

print(f"Before deduplication: {before_dedup}")
print(f"After deduplication: {after_dedup}")
print(f"Removed {before_dedup - after_dedup} duplicates")

## 9. Save the Augmented Dataset

In [ ]:
# Add UUID for each entry
import uuid
combined_train['oare_id'] = [str(uuid.uuid4()) for _ in range(len(combined_train))]

# Reorder columns
combined_train = combined_train[['oare_id', 'transliteration', 'translation', 'source']]

# Save augmented training data
combined_train.to_csv('train_augmented.csv', index=False)
print("Saved augmented training data to train_augmented.csv")

print(f"\n--- Final Dataset Summary ---")
print(f"Total samples: {len(combined_train)}")
print(f"\nSource distribution:")
print(combined_train['source'].value_counts())
print(f"\nData increase from original: {len(combined_train)/len(train):.1f}x")